In [33]:
from datasets import load_dataset

In [34]:
papers_data = load_dataset('csv', data_files='./data/redacoes.csv')

In [35]:
papers_data

DatasetDict({
    train: Dataset({
        features: ['essay', 'score'],
        num_rows: 4570
    })
})

In [36]:
papers_data['train'].features

{'essay': Value('large_string'), 'score': Value('int64')}

In [37]:
papers_data['train'].to_pandas()

,essay,score
0,A convivência com o ensino em uma escola públi...,4
1,O Brasil possui aproximadamente 425 mil polici...,5
2,"O esporte é muito importante, as Olimpíadas, u...",8
3,"O conceito de ""banalidade do mal"" da filósofa ...",8
4,Um dos assuntos que mais geram discussões atua...,5
...,...,...
4565,"Com o início da era digital, a capacidade de t...",6
4566,"O feminicídio é o assassinato de uma mulher, a...",4
4567,Foram vrias as conquistas que as mulheres bras...,5
4568,Segundo o psicoterapeuta Ivan Carpelatto: “a v...,9


In [38]:
train_test = papers_data['train'].train_test_split(test_size=0.2, shuffle=False)
train_test

DatasetDict({
    train: Dataset({
        features: ['essay', 'score'],
        num_rows: 3656
    })
    test: Dataset({
        features: ['essay', 'score'],
        num_rows: 914
    })
})

In [39]:
from datasets import DatasetDict

In [40]:
from transformers import AutoTokenizer
base_model = 'Geotrend/distilbert-base-pt-cased'
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [41]:
def tokenizer_data(text_data):
    return tokenizer(text_data['essay'], truncation=True)

In [42]:
papers_data

DatasetDict({
    train: Dataset({
        features: ['essay', 'score'],
        num_rows: 4570
    })
})

In [43]:
tokenized_dataset = train_test.map(tokenizer_data, batched=True, remove_columns=['essay'])

In [44]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['score', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3656
    })
    test: Dataset({
        features: ['score', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 914
    })
})

In [45]:
from datasets import ClassLabel

In [46]:
scores = ClassLabel(names=[str(i) for i in range(11)])

In [47]:
def map_labels(data):
    data['label'] = scores.str2int(str(data['score']))
    return data

In [48]:
tokenized_dataset = tokenized_dataset.map(map_labels, remove_columns=['score'])

In [49]:
tokenized_dataset['train'].features['label']

Value('int64')

In [50]:
tokenized_dataset = tokenized_dataset.cast_column('label', scores)

In [51]:
tokenized_dataset['train'].features['label']

ClassLabel(names=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10'])